# Przegląd Playbooków – Threat Hunting Lab

Notebook do przeglądania playbooków i przygotowania zapytań **na sucho** – bez połączenia z ELK, MS Defender ani bazą danych.

Użycie:
1. Uruchom komórki sekwencyjnie
2. Przeglądaj listę playbooków, metadane (hipoteza, wymagania środowiska, kroki operacyjne)
3. Wybierz **klasę narzędzia** (siem, edr) – analityk wybiera dostępne narzędzie
4. Kopiuj zapytania do MS Defender Advanced Hunting, Elasticsearch lub innego SIEM/EDR

## 1. Konfiguracja – ścieżka do projektu

In [ ]:
import sys
import importlib
from pathlib import Path

# Auto-detect project root (cwd lub parent jeśli cwd=notebooks/)
cwd = Path.cwd()
if (cwd / "playbooks").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "playbooks").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path("/home/user/Desktop/TH/th_timmy")  # fallback – dostosuj
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import z cli_helpers (get_queries_resolved ma parametr tool_class)
from automation_scripts.playbooks import list_playbooks, show_playbook, validate_playbook
from automation_scripts.playbooks.cli_helpers import get_queries_resolved

# Odśwież moduły po zmianach w kodzie (Kernel → Restart jeśli nadal błędy)
importlib.invalidate_caches()

print(f"Projekt: {PROJECT_ROOT}")

## 2. Lista playbooków

In [ ]:
playbooks_dir = PROJECT_ROOT / "playbooks"
items = list_playbooks(playbooks_dir=playbooks_dir, include_template=False)

for item in items:
    print(f"{item['id']:<35} {item.get('mitre_technique_id', '-'):<8} {item['name']}")

## 3. Szczegóły playbooka

In [ ]:
PLAYBOOK_ID = "T1055-process-injection"  # zmień na wybrany playbook

meta = show_playbook(PLAYBOOK_ID, playbooks_dir=playbooks_dir)

print("=== Metadane ===")
for key in ["name", "mitre_technique_id", "mitre_technique_name", "description", "author", "version"]:
    if key in meta and meta[key]:
        print(f"\n{key}:\n{meta[key]}")

print("\n=== Wymagania środowiska (klasy narzędzi) ===")
env = meta.get("environment_requirements", {})
if env:
    print("Klasy narzędzi:", env.get("tool_classes", []))
    print(env.get("tool_class_descriptions", ""))
else:
    print("(brak)")

print("\n=== Hipoteza ===")
hyp = meta.get("hypothesis", {})
if hyp:
    for k, v in hyp.items():
        if v:
            print(f"  {k}: {v}")
else:
    print("(brak)")

print("\n=== Opis techniki ===")
td = meta.get("technique_description", "")
print(td[:500] + "..." if len(td) > 500 else td)

print("\n=== Kroki operacyjne ===")
steps = meta.get("operational_steps", [])
for s in steps[:4]:
    print(f"  Krok {s.get('step')}: {s.get('name')} - {s.get('operation', '')[:60]}...")

print("\n=== Eskalacja ===")
esc = meta.get("escalation", {})
if esc:
    print(f"  Na potwierdzenie: {esc.get('on_confirmed', '')}")
    print(f"  Role: {esc.get('notification_roles', [])}")

## 4. Zapytania (filtr po klasie narzędzia)

Ustaw **TOOL_CLASS** na dostępną klasę: `siem` (ELK, Splunk) lub `edr` (MS Defender). Analityk wybiera klasę i otrzymuje gotowe zapytania dla swojego narzędzia.

In [ ]:
# Wybierz klasę narzędzia: siem | edr | data_lake (None = wszystkie)
TOOL_CLASS = "edr"  # np. "siem" dla ELK/Splunk, "edr" dla MS Defender

try:
    queries = get_queries_resolved(
        PLAYBOOK_ID,
        hours=24,
        playbooks_dir=playbooks_dir,
        tool_class=TOOL_CLASS if TOOL_CLASS else None,
    )
except TypeError as e:
    if "tool_class" in str(e):
        # Starsza wersja – załaduj wszystkie i filtruj ręcznie
        all_q = get_queries_resolved(PLAYBOOK_ID, hours=24, playbooks_dir=playbooks_dir)
        if TOOL_CLASS:
            queries = [q for q in all_q if getattr(q, "tool_class", None) == TOOL_CLASS]
        else:
            queries = all_q
        print("(Uwaga: użyto fallback – zrestartuj kernel dla pełnej obsługi tool_class)\n")
    else:
        raise

print(f"Zapytania dla klasy: {TOOL_CLASS or 'wszystkie'} ({len(queries)} zapytań)\n")
for q in queries:
    tc = getattr(q, "tool_class", None) or "-"
    print(f"\n--- {q.tool} ({tc}) / {q.mode} / {q.query_path} ---")
    print(q.content)

## 5. Walidacja playbooka

In [ ]:
# validate_placeholders=False – playbooki używają czasu względnego (ago(7d)), nie placeholderów
result = validate_playbook(playbooks_dir / PLAYBOOK_ID, validate_placeholders=False)

if result.success:
    print("Playbook poprawny.")
    if result.warnings:
        print("Ostrzeżenia:", result.warnings)
else:
    print("Błędy:", result.errors)

## 6. Uruchomienie CLI z komórki (alternatywa)

In [ ]:
# Możesz też uruchomić skrypt CLI bezpośrednio:
import os
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
!python {PROJECT_ROOT / 'scripts' / 'th_playbook.py'} list